# PyCAM-SIMA persistent Dask

This Notebook runs one long-lived 24-rank CAM model through one Dask Actor. One literal `with experiments.model(...) as model:` block contains the complete demonstration, so every operation uses the same MPI ranks and Python-owned StatePool. Leaving the block closes the model automatically.

## 1. Execution model

```text
Jupyter Notebook
      │
      └── with experiments.model("persistent-base") as model
                         │
                         ▼
                  pinned Dask Actor
                         │ starts once
                         ▼
                  24 persistent MPI ranks
                         │
                         ├── Python CAMDriver
                         ├── Python StatePool
                         └── Fortran device .so
```

In PBS mode the base model submits one PBS job. In allocation mode it starts one MPI world inside the current allocation. Optional persistent forks are disabled by default because every live child requires another PBS job.

## 2. Configure the experiment

In [18]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from pycam_sima import DaskExperimentClient

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = (
    scratch / 'pycam-sima/persistent_notebook_trials' / stamp
)
initial_run_dir = experiment_root / 'initial-run'
run_root = experiment_root / 'models'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'

# False means one base model and one MPI launch. Set this to True only
# when three independent live fork children, and their PBS jobs, are wanted.
run_persistent_fork = False
dask_workers = (
    3 if execution_mode == 'pbs' and run_persistent_fork else 1
)

print('pycam_sima', pycam_sima.__version__)
print('experiment root:', experiment_root)
print('execution mode:', execution_mode)
print('persistent fork enabled:', run_persistent_fork)

pycam_sima 0.15.0
experiment root: /glade/derecho/scratch/ruitong/pycam-sima/persistent_notebook_trials/20260725-024540
execution mode: pbs
persistent fork enabled: False


## 3. Run the complete persistent demonstration

The comments divide this single cell into logical sections. The one `model` remains alive throughout: complete steps, dynamic variables, runtime Fortran plugins, direct scheme/phase calls, and an optional fork all operate on the same StatePool.

In [19]:
# Clean up objects from an earlier execution of this cell.
if 'model_scope' in globals():
    model_scope.close()
    del model_scope
if 'model' in globals():
    try:
        model.close()
    except RuntimeError:
        pass
if 'client' in globals():
    client.close()

# Start the local Dask scheduler/workers. This does not submit a PBS job.
client = Client(
    processes=False,
    n_workers=dask_workers,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=run_root,
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)

try:
    # This is the only model creation in the Notebook. In PBS mode it
    # submits one job and starts 24 MPI ranks; all code below reuses them.
    with experiments.model('persistent-base') as model:
        # --------------------------------------------------------------
        # A. Inspect the initialized StatePool and run two complete steps.
        # --------------------------------------------------------------
        started = model.status
        temperature_before = model.fields.air_temperature.stats(rank=0)

        model.advance(steps=2)

        temperature_after = model.fields.air_temperature.stats(rank=0)
        checkpoint = model.save()
        after_two_steps = model.status

        assert started.mpi_launch_count == 1
        assert after_two_steps.mpi_launch_count == 1
        assert after_two_steps.worker_pid == started.worker_pid
        assert after_two_steps.step == started.step + 2

        # --------------------------------------------------------------
        # B. Dynamically allocate a new Python-owned StatePool variable.
        #    It is created collectively on all 24 existing MPI ranks.
        # --------------------------------------------------------------
        model.fields.create(
            'experiment_tracer',
            dims=('column', 'level'),
            units='kg kg-1',
            initial=0.0,
        )
        tracer_stats = model.fields.experiment_tracer.stats(rank=0)
        after_variable = model.status

        assert after_variable.worker_pid == started.worker_pid
        assert tracer_stats['mean'] == 0.0

        # --------------------------------------------------------------
        # C. Dynamically build, load, place, and run an original-Fortran
        #    physics function without restarting MPI or the StatePool.
        # --------------------------------------------------------------
        installed = model.physics.install(
            source=(
                repo
                / 'examples/plugins/runtime_temperature_offset/device.yaml'
            ),
            project_root=repo,
            after='kessler',
            inputs={
                'runtime_plugin_temperature': 240.0,
                'runtime_plugin_temperature_increment': 1.5,
            },
        )
        plugin_temperature = (
            model.fields.ccpp_runtime_plugin_temperature
        )
        before_plugin = plugin_temperature.stats(rank=0)

        model.physics.scheme(
            'runtime_temperature_offset', group='before'
        ).run()

        after_plugin = plugin_temperature.stats(rank=0)
        after_plugin_call = model.status

        assert after_plugin_call.worker_pid == started.worker_pid
        assert np.isclose(
            after_plugin['mean'] - before_plugin['mean'], 1.5
        )

        # A device.yaml may export several functions through several
        # entrypoints/processes. Use PhysicsPluginSpec with multiple
        # SchemePlacement entries when those functions need different
        # positions in the live execution plan.

        # --------------------------------------------------------------
        # D. Run one scheme and one phase explicitly. Neither advances
        #    model time. Then run one complete step, which does advance it.
        # --------------------------------------------------------------
        before_control = model.status

        model.physics.kessler.run()
        after_scheme = model.status

        model.phases.physics_to_dynamics.run()
        after_phase = model.status

        model.advance(steps=1)
        after_complete_step = model.status

        assert after_scheme.step == before_control.step
        assert after_phase.step == before_control.step
        assert after_complete_step.step == before_control.step + 1
        assert after_complete_step.worker_pid == started.worker_pid
        assert after_complete_step.mpi_launch_count == 1

        # --------------------------------------------------------------
        # E. Optionally fork three independent persistent children from
        #    this exact in-memory StatePool. Disabled by default because
        #    every child starts a separate 24-rank PBS job.
        # --------------------------------------------------------------
        if not run_persistent_fork:
            fork_result = 'skipped (run_persistent_fork=False)'
        elif execution_mode != 'pbs':
            fork_result = 'skipped (persistent fork requires PBS mode)'
        else:
            control = experiments.plan('memory-control')
            no_kessler = experiments.plan(
                'memory-no-kessler', experimental=True
            )
            no_kessler.physics.kessler.disable()
            warm = experiments.plan('memory-warm')
            warm.fields.edit('air_temperature', 'add', 1.0)

            base_temperature = model.fields.air_temperature.get(rank=0)
            children = experiments.fork_models(
                model,
                (control, no_kessler, warm),
                close_parent=False,
            )
            with children:
                initial_temperature = {
                    name: child.fields.air_temperature.get(rank=0)
                    for name, child in children.items()
                }
                assert np.array_equal(
                    initial_temperature['memory-control'],
                    base_temperature,
                )
                assert np.array_equal(
                    initial_temperature['memory-no-kessler'],
                    base_temperature,
                )
                assert np.array_equal(
                    initial_temperature['memory-warm'],
                    np.add(base_temperature, 1.0),
                )
                children.advance(steps=1)
                fork_result = children.statuses

        # Save lightweight values for display after the context closes.
        final_status = model.status
        result = {
            'model': final_status.name,
            'pbs_job_id': final_status.pbs_job_id,
            'worker_pid': final_status.worker_pid,
            'mpi_launch_count': final_status.mpi_launch_count,
            'step': final_status.step,
            'temperature_before': temperature_before,
            'temperature_after_two_steps': temperature_after,
            'dynamic_variable': tracer_stats,
            'installed_plugin': installed['plugin']['name'],
            'plugin_temperature_before': before_plugin,
            'plugin_temperature_after': after_plugin,
            'checkpoint': str(checkpoint.path),
            'fork': fork_result,
        }
finally:
    # Leaving the with block closes the Actor and all MPI ranks. The
    # Dask scheduler/workers are separate and are closed here as well.
    client.close()

result

PyCAM-SIMA PBS worker submitted as 6902399.desched1; waiting for 24 MPI ranks ...


{'model': 'persistent-base',
 'pbs_job_id': '6902399.desched1',
 'worker_pid': 64490,
 'mpi_launch_count': 1,
 'step': 3,
 'temperature_before': {'rank': 0,
  'shape': (4, 4, 30, 3, 3),
  'dtype': '<f8',
  'min': 149.84337724047597,
  'max': 306.20463514548004,
  'mean': 237.25174410239765},
 'temperature_after_two_steps': {'rank': 0,
  'shape': (4, 4, 30, 3, 3),
  'dtype': '<f8',
  'min': 149.84300817997197,
  'max': 306.21770949209105,
  'mean': 237.2520353018612},
 'dynamic_variable': {'rank': 0,
  'shape': (27, 30),
  'dtype': '<f8',
  'min': 0.0,
  'max': 0.0,
  'mean': 0.0},
 'installed_plugin': 'runtime_temperature_offset',
 'plugin_temperature_before': {'rank': 0,
  'shape': (27, 30),
  'dtype': '<f8',
  'min': 240.0,
  'max': 240.0,
  'mean': 240.0},
 'plugin_temperature_after': {'rank': 0,
  'shape': (27, 30),
  'dtype': '<f8',
  'min': 241.5,
  'max': 241.5,
  'mean': 241.5},
 'checkpoint': '/glade/derecho/scratch/ruitong/pycam-sima/persistent_notebook_trials/20260725-024540